# 15.11 k732. 特殊位置（APCS 2023-06 實作第 2 題 / 官方中級題本範例第 1 題）

| 屬性 | 規格說明 |
| :--- | :--- |
| **適合對象** | 程式設計初學者（完全零基礎） / APCS 扎根學習者 |
| **前置核心語法** | 二維陣列走訪、絕對值函數 `abs()`、自訂函式、邊界保護條件、模運算 `%`、字典序排序 |
| **學習目標** | 掌握曼哈頓距離幾何模型、菱形走訪半徑推導與行範圍縮小法、邊界安全防護、特殊位置判定與自然字典序輸出 |
| **教材對應** | 官方中級題本範例第 1 題（ZeroJudge k732） / APCS 2023 年 6 月實作第 2 題 |
| **難度評級** | ⭐⭐⭐☆☆（中級題經典 / 二維陣列曼哈頓幾何搜尋） |

---

### 💡 單元導言：官方指標試題——二維幾何與範圍搜尋的經典代表作
各位程式冒險者好！歡迎來到第十五章「APCS 實作真題特訓（中級題）」的精彩篇章。
本單元所選取的試題 **k732. 特殊位置**，是 2023 年 6 月 APCS 實作考題，同時也是教育部官方發布的《APCS 程式實作中級題本範例》中的**第一道指標範例試題**！這意味著本題在檢定官方眼中，代表了檢驗考生是否具備「中級程式設計與演算法能力」的最標準度量衡。

本題的核心挑戰在於二維網格中的「曼哈頓幾何搜尋」：每個格子以自身數值作為半徑，向外擴散形成一個 45 度旋轉的菱形區域，並計算區域內有效格子的數值總和是否滿足特定餘數條件。
初學同學常遇到的痛點包括：
1. **幾何邊界溢出**：菱形向外擴散時容易超出網格邊界導致 `IndexError`。
2. **多重迴圈效能盲區**：若無策略地對全圖暴力搜尋，容易造成多餘運算。
3. **輸出排序要求**：題目要求依列標與行標字典序輸出，如何運用 Python 自然走訪特性免除額外排序。

為了引導各位學習者無痛征服這道官方標竿大題，我們規劃了「8 大極致緩坡學習階梯」，從生活中的曼哈頓計程車幾何模型出發，一步步推導行範圍縮小公式、邊界鉗制防護、餘數判定，最後完成 100% 滿分 AC 代碼！

---

### 🗺️ 8 大漸進式學習階梯導航地圖

```
┌────────────────────────────────────────────────────────────────────────┐
│               15.11 特殊位置（ZeroJudge k732）8 階梯學習路徑             │
├────────────────────────────────────────────────────────────────────────┤
│ 🪜 階梯 1：題意解析與曼哈頓距離幾何模型（菱形覆蓋半徑 |i-s|+|j-t| ≤ x） │
│ 🪜 階梯 2：網格走訪基本功：列行雙重迴圈與中心點 A[i][j] 數值讀取        │
│ 🪜 階梯 3：菱形區域求和演算法（一）：暴力法枚舉全圖點與絕對值不等式篩選 │
│ 🪜 階梯 4：菱形區域求和演算法（二）：行範圍縮小法與邊界防護（核心效能躍升）│
│ 🪜 階梯 5：特殊位置判定條件：元素和模 10 與中心值模 10 之餘數比對     │
│ 🪜 階梯 6：多解座標收集與自然字典序排序（先比列標 i 再比行標 j）       │
│ 🪜 階梯 7：完整 AC 模組組裝：雙子題組（n=1 一維特例與 50x50 二維）通關程式│
│ 🪜 階梯 8：極端邊界壓力測試（單點、菱形溢出、無解）、複雜度與 WA 排查   │
├────────────────────────────────────────────────────────────────────────┤
│ 🏆 結尾附錄：雙平台滿分通關解答庫（淺顯一般版 / 極簡高效版 / ZeroJudge 萬用版）│
└────────────────────────────────────────────────────────────────────────┘
```

## 📜【APCS 官方完整真題題面與規範】

### 1. 題目資訊快覽
* **題目名稱**：特殊位置（Special Position）
* **歷屆出處**：APCS 2023 年 6 月場次實作第 2 題 / 教育部官方中級題本範例第 1 題
* **線上評判**：ZeroJudge k732
* **難度評級**：⭐⭐⭐☆☆（中級題）
* **測資規範**：每一筆測試資料執行時間限制均為 1.0 秒，空間限制 256 MB，保證所有測資皆符合輸入規格。

---

### 2. 完整題目描述（Problem Description）
考慮一個二維正整數陣列 $A$，有 $n$ 列 (row) 及 $m$ 行 (column)。陣列 $A$ 中第 $i$ 列第 $j$ 行的元素為 $A[i][j]$。兩個元素 $A[i][j]$ 與 $A[s][t]$ 的距離定義為曼哈頓距離：
$$| i - s | + | j - t |$$
也就是列註標差的絕對值加上行註標差的絕對值。

對於一個元素 $A[i][j] = x$，若與其**距離不超過 $x$ 的所有元素總和（包含 $A[i][j]$ 本身）除以 10 的餘數**，與 **$x$ 除以 10 的餘數相同**，則我們稱 $(i, j)$ 為一**特殊位置**。
> ⚠️ **注意**：在計算此總和時，超出陣列範圍的元素不予計算。

今給定一陣列，你的任務要找出陣列 $A$ 中特殊位置的總數以及每個特殊位置；一個位置以兩個整數表示，分別代表其列註標與行註標。

舉例來說，考慮下列 $5 	imes 6$ 陣列 $A$：

```text
           直行 (col)
        0   1   2   3   4   5
      ┌───┬───┬───┬───┬───┬───┐
横  0 │ 1 │ 3 │ 4 │[1]│ 3 │ 1 │
列  1 │ 1 │ 1 │[4]│[1]│[3]│ 1 │
(row)2 │ 1 │[1]│[3]│[2]│[5]│[3]│  ← 中心 A[2][3]=2 (曼哈頓距離 ≤ 2 範圍以 [ ] 標記)
    3 │ 4 │ 3 │[3]│[1]│[4]│ 1 │
    4 │ 5 │ 2 │ 1 │[1]│ 1 │ 1 │
      └───┴───┴───┴───┴───┴───┘
```

* 其中 $A[0][0] = 1$，與其距離 1 以內的元素只有三個：其本身、右方的 3 與下方的 1，其總和為 $1 + 3 + 1 = 5$。5 除以 10 的餘數為 5，與 $A[0][0]=1$ 除以 10 的餘數不同，故 $(0, 0)$ 並非特殊位置。
* 而 $A[2][3] = 2$，與其距離 2 以內有 13 個元素，如上圖陰影處（以 `[ ]` 標記）所示，其元素總和為：
  $$1 + (4 + 1 + 3) + (1 + 3 + 2 + 5 + 3) + (3 + 1 + 4) + 1 = 32$$
  32 除以 10 的餘數為 2，與 $A[2][3]=2$ 除以 10 的餘數（$2 \pmod{10} = 2$）相同，故 $(2, 3)$ 為一特殊位置。
* 此例中特殊位置共有 4 個，分別為 $(0, 2)$、$(1, 3)$、$(2, 3)$ 以及 $(3, 3)$。

---

### 3. 輸入說明（Input Format）
* 第一行有兩個正整數 $n$ 與 $m$（$1 \le n, m \le 50$），代表陣列的列數與行數。
* 接下去有 $n$ 行，每行有 $m$ 個整數，表示陣列的內容，其順序為由上而下，由左至右。
* 同一行的兩數字之間以一個半形空白間隔。
* 陣列內的數字均為小於 10 的正整數（即 $1 \le A[i][j] \le 9$）。

---

### 4. 輸出說明（Output Format）
* 第一行輸出一個整數 $k$，表示所求之特殊位置總數。
* 接下來 $k$ 行，每行有兩個整數，依序表示一個特殊位置的列註標與行註標。
* **排序規則**：輸出時，列註標較小的特殊位置先輸出；若兩個特殊位置的列註標相同，則先輸出行註標較小者（即標準字典序）。
* 若沒有任何特殊位置（$k = 0$），則第一行輸出 `0`，後續不需輸出任何座標。

---

### 5. 官方完整範例測資一覽表

| 測資編號 | 輸入內容 | 預期輸出 | 幾何推導與判定說明 |
| :--- | :--- | :--- | :--- |
| **官方範例一**<br>（子題組 1：$n=1$ 一維特例） | `1 5`<br>`3 1 4 5 1` | `2`<br>`0 0`<br>`0 2` | 一維陣列 $(0, 0)=3$，距離 $\le 3$ 涵蓋索引 0~3，總和 $3+1+4+5=13$，$13 \pmod{10}=3=3$，符合！<br>$(0, 2)=4$，距離 $\le 4$ 涵蓋全部 5 格，總和 $14$，$14 \pmod{10}=4=4$，符合！其餘不符。總計 2 個。 |
| **官方範例二**<br>（完整二維矩陣） | `5 6`<br>`1 3 4 1 3 1`<br>`1 1 4 1 3 1`<br>`1 1 3 2 5 3`<br>`4 3 3 1 4 1`<br>`5 2 1 1 1 1` | `4`<br>`0 2`<br>`1 3`<br>`2 3`<br>`3 3` | $(0, 2)=4$ 菱形和為 24，$24 \pmod{10}=4$ 符合；<br>$(1, 3)=1$ 菱形和為 11，$11 \pmod{10}=1$ 符合；<br>$(2, 3)=2$ 菱形和為 32，$32 \pmod{10}=2$ 符合；<br>$(3, 3)=1$ 菱形和為 11，$11 \pmod{10}=1$ 符合。共 4 個。 |
| **邊界測資一**<br>（極限單點 $1 	imes 1$） | `1 1`<br>`3` | `1`<br>`0 0` | 僅有單格 $(0, 0)=3$，距離 $\le 3$ 僅包含自身，總和為 3。$3 \pmod{10}=3$，符合條件！輸出 1 與 `0 0`。 |
| **邊界測資二**<br>（無符合特殊位置） | `1 2`<br>`2 5` | `0` | $(0, 0)=2$，距離 $\le 2$ 總和 $2+5=7 
e 2$；<br>$(0, 1)=5$，距離 $\le 5$ 總和 $2+5=7 
e 5$。無特殊位置，輸出 `0`。 |

---

### 6. 評分說明與測資範圍
* **第 1 子題組（60分）**：$n = 1$（一維陣列特例，僅有一橫列）。
* **第 2 子題組（40分）**：無額外限制（$1 \le n, m \le 50$，完整二維矩陣）。

## 15.11.1 題意解析與曼哈頓距離幾何模型（菱形覆蓋半徑 $|i - s| + |j - t| \le x$）

### 🎯 幾何心智模型：計程車街區與 45 度旋轉菱形
在日常幾何學中，我們最熟悉的距離是「歐幾里得距離（直線距離）」，即兩點之間的直線長度 $\sqrt{(i-s)^2 + (j-t)^2}$。
然而在棋盤、網格地圖或城市街道中，我們無法穿牆走斜線，只能沿著橫向與縱向街區行走。這種只能「水平走」與「垂直走」所累積的步數，在數學上被稱為**曼哈頓距離（Manhattan Distance）**，又稱**計程車幾何（Taxicab Geometry）**。

其數學公式極為簡潔：
$$d((i, j), (s, t)) = |i - s| + |j - t|$$

當我們設定一個中心點 $(i, j)$，並要求所有與其曼哈頓距離不大於 $x$ 的點 $(s, t)$ 時，在幾何圖形上會形成一個**旋轉 45 度的正方形（即菱形區域）**：
* 菱形最上方的頂點為 $(i - x, j)$
* 菱形最下方的頂點為 $(i + x, j)$
* 菱形最左方的頂點為 $(i, j - x)$
* 菱形最右方的頂點為 $(i, j + x)$

```text
                  (i-2, j)             ← 列差 dr = -2, 行差 dc = 0
                     │
          (i-1, j-1)─┼─(i-1, j+1)      ← 列差 dr = -1, 行差 dc ∈ [-1, 1]
               │     │     │
  (i, j-2)─────┼──(i, j)──┼─────(i, j+2)← 列差 dr = 0, 行差 dc ∈ [-2, 2]
               │     │     │
          (i+1, j-1)─┼─(i+1, j+1)      ← 列差 dr = +1, 行差 dc ∈ [-1, 1]
                     │
                  (i+2, j)             ← 列差 dr = +2, 行差 dc = 0
```

### ⚠️ 初學者常見思維盲點
1. **混淆歐氏距離**：切勿使用根號或浮點數運算。曼哈頓距離純粹是整數絕對值相加，計算極快且精準無誤差。
2. **包含中心自身**：中心點 $(i, j)$ 與自身的曼哈頓距離為 $|i-i| + |j-j| = 0 \le x$，因此中心點自身**必然包含在總和範圍內**，絕不可漏算！
3. **出界點不予計算**：若菱形的頂點超出網格邊界（例如 $i - x < 0$），題目明確規範「超出陣列範圍的元素不予計算」，因此只需累加網格內的格子即可。

讓我們透過以下程式碼，實作曼哈頓距離計算函式，並直觀觀察兩點距離判定。

In [ ]:
# 15.11.1 範例展示：曼哈頓距離計算與菱形半徑判定
def manhattan_dist(r1, c1, r2, c2):
    """計算兩座標點 (r1, c1) 與 (r2, c2) 的曼哈頓距離"""
    return abs(r1 - r2) + abs(c1 - c2)

# 假設中心點位於 (2, 3)，半徑 x = 2
center_r, center_c = 2, 3
radius_x = 2

# 測試四個不同位置的點
test_points = [
    (2, 3),  # 中心點自身
    (1, 2),  # 距離 = |2-1| + |3-2| = 1 + 1 = 2 (在菱形邊界上)
    (0, 3),  # 距離 = |2-0| + |3-3| = 2 + 0 = 2 (在菱形正上方頂點)
    (0, 0),  # 距離 = |2-0| + |3-0| = 2 + 3 = 5 (超出菱形範圍)
]

print(f"中心點：({center_r}, {center_c})，半徑 x = {radius_x}")
print("-" * 45)
for pr, pc in test_points:
    d = manhattan_dist(center_r, center_c, pr, pc)
    is_inside = (d <= radius_x)
    print(f"目標點 ({pr}, {pc}) -> 距離 = {d:2d} -> 是否在菱形內：{is_inside}")

In [ ]:
# 填空 15.11.1：判斷目標座標是否落在中心點的曼哈頓菱形覆蓋半徑內
# 請將 ___ 替換為適當的運算式或變數

def is_within_diamond(center_r, center_c, target_r, target_c, x):
    # 步驟 1：計算列座標差的絕對值加上行座標差的絕對值
    distance = abs(center_r - target_r) + abs(center_c - target_c)
    
    # 步驟 2：判斷距離是否不超過 x
    return distance <= ___

# 測試填空結果
print("驗證 (2, 3) 與 (3, 4) 在半徑 2 內：", is_within_diamond(2, 3, 3, 4, 2))
print("驗證 (2, 3) 與 (4, 5) 在半徑 2 內：", is_within_diamond(2, 3, 4, 5, 2))

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.1：候選座標清單之菱形半徑過濾器
# 任務說明：給定中心座標 (center_r, center_c)、半徑 x 與一組座標清單 candidates。
# 請篩選出所有與中心點曼哈頓距離 <= x 的座標，並依原順序回傳。
#
# 【公開測試資料 1】
# 中心點：(2, 3)，半徑 x = 2，候選座標：[(2, 3), (1, 2), (0, 0), (4, 3), (3, 5)]
# 預期輸出：[(2, 3), (1, 2), (4, 3)]
#
# 【公開測試資料 2】
# 中心點：(0, 0)，半徑 x = 1，候選座標：[(0, 1), (1, 0), (1, 1), (2, 0)]
# 預期輸出：[(0, 1), (1, 0)]
# ==========================================
def filter_within_diamond(center_r, center_c, x, candidates):
    # 請在下方撰寫你的程式碼：
    result = []
    for r, c in candidates:
        if abs(center_r - r) + abs(center_c - c) <= x:
            result.append((r, c))
    return result

# 執行測試驗證
print("測試 1 結果：", filter_within_diamond(2, 3, 2, [(2, 3), (1, 2), (0, 0), (4, 3), (3, 5)]))
print("測試 2 結果：", filter_within_diamond(0, 0, 1, [(0, 1), (1, 0), (1, 1), (2, 0)]))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.1：菱形邊緣環狀格子計數
# 任務說明：在一個 N x M 的網格中（座標範圍 0 <= r < N, 0 <= c < M），
# 給定中心點 (cr, cc) 與整數距離 d。
# 請計算出網格內與中心點曼哈頓距離「恰好等於 d」的合法格子總數。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
def count_exact_manhattan_boundary(n, m, cr, cc, d):
    # 請在此處撰寫你的程式碼：
    count = 0
    for r in range(n):
        for c in range(m):
            if abs(r - cr) + abs(c - cc) == d:
                count += 1
    return count

# 快速自我檢驗：在 5x5 網格中，中心 (2, 2) 距離恰好為 1 的點應有 4 個 (上、下、左、右)
print("中心 (2, 2) 距離為 1 的格子數：", count_exact_manhattan_boundary(5, 5, 2, 2, 1))

## 15.11.2 網格走訪基本功：列行雙重迴圈與中心點 $A[i][j]$ 數值讀取

### 🎯 網格走訪的心智模型：逐列掃描雷達
在處理所有二維陣列（2D Matrix / Grid）相關的競賽試題時，最基礎且不可或缺的內功就是「標準雙重迴圈走訪」。
想像一架雷達螢幕：
1. **外層迴圈**：控制垂直方向的**「列指標（Row Index）」** $i$，從最頂端的第 0 列依序往下移動到第 $n-1$ 列（`range(n)`）。
2. **內層迴圈**：控制水平方向的**「行指標（Column Index）」** $j$，從最左側的第 0 行依序往右掃描到第 $m-1$ 行（`range(m)`）。
3. **元素存取**：當指標落在 $(i, j)$ 時，透過 `grid[i][j]` 讀取該格子的數值 $x$。在本作中，這個 $x$ 將同時扮演兩大角色：
   * **角色一**：作為曼哈頓搜尋半徑的上限值。
   * **角色二**：作為最後餘數比對的基準目標值。

### ⚠️ 考場常見 IndexError 避坑指南
初學者在手寫網格迴圈時，最常發生的低級失誤是**「行列變數顛倒」**：
* 誤寫成 `for i in range(m): for j in range(n): ... grid[i][j]`。當矩陣不是正方形（例如 $n = 3, m = 8$）時，當 $i$ 跑到 7 就會立刻觸發崩潰性的 `IndexError: list index out of range`！
* **黃金防禦法則**：永遠將變數命名語意化，外層使用 `r`（Row）或 `i` 對應 `range(n)`；內層使用 `c`（Col）或 `j` 對應 `range(m)`；存取時永遠遵循 `grid[r][c]`（先行後列是直角座標系，但矩陣索引永遠是**「先列後行」**！）。

讓我們透過以下範例，熟悉二維網格的標準走訪與即時數值解鎖。

In [ ]:
# 15.11.2 範例展示：二維網格雙重迴圈走訪與中心點數值萃取
# 建立一個 3x4 的測試矩陣
grid = [
    [3, 1, 4, 2],
    [5, 9, 2, 6],
    [1, 7, 3, 8]
]
n = len(grid)     # 列數 = 3
m = len(grid[0])  # 行數 = 4

print(f"網格規格：{n} 列 x {m} 行")
print("-" * 35)

# 標準雙重迴圈逐格走訪
for r in range(n):
    for c in range(m):
        x = grid[r][c]
        print(f"座標 ({r}, {c}) 的中心數值 x = {x}")

In [ ]:
# 填空 15.11.2：走訪 n x m 矩陣並統計偶數元素個數
# 請將 ___ 替換為適當的變數或運算式

def count_even_numbers(grid, n, m):
    even_count = 0
    # 外層走訪列 (0 到 n-1)
    for i in range(___):
        # 內層走訪行 (0 到 m-1)
        for j in range(___):
            val = grid[i][___]
            if val % 2 == 0:
                even_count += 1
    return even_count

test_grid = [
    [1, 2, 3],
    [4, 5, 6]
]
print("偶數個數（應為 3）：", count_even_numbers(test_grid, 2, 3))

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.2：網格特定門檻元素座標搜尋
# 任務說明：給定 n x m 的矩陣 grid 與門檻值 threshold。
# 請由上而下、由左至右走訪網格，找出所有「數值 >= threshold」的格子，
# 將其 (列座標, 行座標, 數值) 組成三元組串列並回傳。
#
# 【公開測試資料 1】
# grid = [[1, 6, 2], [5, 3, 8], [4, 9, 0]], threshold = 5
# 預期輸出：[(0, 1, 6), (1, 0, 5), (1, 2, 8), (2, 1, 9)]
#
# 【公開測試資料 2】
# grid = [[7, 2, 3, 5], [1, 9, 4, 2]], threshold = 5
# 預期輸出：[(0, 0, 7), (0, 3, 5), (1, 1, 9)]
# ==========================================
def find_elements_above_threshold(grid, threshold):
    # 請在下方撰寫你的程式碼：
    n = len(grid)
    m = len(grid[0])
    res = []
    for r in range(n):
        for c in range(m):
            if grid[r][c] >= threshold:
                res.append((r, c, grid[r][c]))
    return res

# 執行測試驗證
print("測試 1 結果：", find_elements_above_threshold([[1, 6, 2], [5, 3, 8], [4, 9, 0]], 5))
print("測試 2 結果：", find_elements_above_threshold([[7, 2, 3, 5], [1, 9, 4, 2]], 5))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.2：網格四邊界元素總和計算
# 任務說明：給定一個 n x m 的二維陣列 grid。
# 請走訪整個網格，計算出所有「位於最外圍邊界上」的格子總和。
# （邊界格子指：r == 0 或 r == n-1 或 c == 0 或 c == m-1）
# （本題為自由挑戰題，無公開測資，請自行構思完整程式碼）
# ==========================================
def sum_grid_borders(grid):
    # 請在此處撰寫你的程式碼：
    n = len(grid)
    m = len(grid[0])
    total = 0
    for r in range(n):
        for c in range(m):
            if r == 0 or r == n - 1 or c == 0 or c == m - 1:
                total += grid[r][c]
    return total

# 快速自我檢驗：3x3 全 1 矩陣，邊界有 8 格，總和應為 8
print("3x3 全 1 矩陣邊界和：", sum_grid_borders([[1]*3 for _ in range(3)]))

## 15.11.3 菱形區域求和演算法（一）：暴力法枚舉全圖點與絕對值不等式篩選

### 🎯 初學者直覺思路：全地圖大清查
當題目要求我們計算「距離中心 $(i, j)$ 不超過 $x$ 的所有格子總和」時，最平鋪直敘、最不易出錯的演算法直覺是什麼？
**「既然不知道菱形長什麼樣子，那我就把地圖上的每一個格子全部看一遍！」**

這就是經典的**全圖暴力枚舉法（Brute Force Scanning）**：
1. 針對固定的中心點 $(i, j)$，宣告一個累加變數 `diamond_sum = 0`。
2. 啟動另外一組雙重迴圈，遍歷地圖上**所有的格子** $(s, t)$，其中 $0 \le s < n$ 且 $0 \le t < m$。
3. 對於每一個 $(s, t)$，透過曼哈頓距離公式計算其與中心的距離：
   $$dist = |i - s| + |j - t|$$
4. 只要 $dist \le x$，就將 `grid[s][t]` 加入 `diamond_sum`。
5. 遍歷完所有點後，`diamond_sum` 就是該中心點對應的菱形區域元素總和！

### 🔍 演算法複雜度與適用邊界評估
讓我們來嚴格計算此方法的運算量：
* 單一中心點需檢查 $n 	imes m$ 個格子。
* 全圖共有 $n 	imes m$ 個格子都需要扮演中心點。
* 因此總時間複雜度為 $O((n 	imes m)^2) = O(n^2 \cdot m^2)$。

在本作的測資範圍中：$n, m \le 50$，故 $n 	imes m \le 2500$。
$2500 	imes 2500 pprox 6.25 	imes 10^6$（約 6 百萬次基本運算）。
在 Python 直譯環境中，每秒約可執行 $10^7$ 次運算。因此 6 百萬次運算在 1.0 秒的時間限制內**剛好可以安全通過**！
這告訴我們：在考場上若一時想不出巧妙的幾何優化，**先寫出暴力法拿下分數是極具戰略價值的明智之舉**！

讓我們透過程式碼實作這套最穩健的暴力求和函式。

In [ ]:
# 15.11.3 範例展示：全圖暴力枚舉菱形區域元素總和
def get_diamond_sum_brute(grid, n, m, i, j, x):
    """
    使用全圖暴力法計算中心 (i, j) 曼哈頓距離 <= x 的元素總和
    時間複雜度：O(n * m)
    """
    diamond_sum = 0
    # 枚舉網格中所有的點 (s, t)
    for s in range(n):
        for t in range(m):
            # 計算與中心的曼哈頓距離
            dist = abs(i - s) + abs(j - t)
            # 若在半徑 x 內，則納入總和
            if dist <= x:
                diamond_sum += grid[s][t]
    return diamond_sum

# 以官方範例二的矩陣進行驗證
grid_sample = [
    [1, 3, 4, 1, 3, 1],
    [1, 1, 4, 1, 3, 1],
    [1, 1, 3, 2, 5, 3],
    [4, 3, 3, 1, 4, 1],
    [5, 2, 1, 1, 1, 1]
]
n, m = len(grid_sample), len(grid_sample[0])

# 測試中心點 (2, 3)，其數值 x = 2
center_r, center_c = 2, 3
x = grid_sample[center_r][center_c]
total = get_diamond_sum_brute(grid_sample, n, m, center_r, center_c, x)

print(f"中心 ({center_r}, {center_c}) 數值 x = {x}")
print(f"暴力求和結果：{total}（官方真題說明預期為 32）")

In [ ]:
# 填空 15.11.3：補齊全圖暴力法篩選與累加邏輯
# 請將 ___ 替換為適當的運算式或變數

def compute_brute_sum(grid, n, m, cr, cc, max_dist):
    total = 0
    for r in range(n):
        for c in range(m):
            # 判斷點 (r, c) 與 (cr, cc) 的距離是否小於等於 max_dist
            if abs(r - cr) + abs(c - cc) <= ___:
                total += grid[___][___]
    return total

test_grid = [[3, 1, 4, 5, 1]]
print("測試一維陣列 (0, 0) 半徑 3 的總和（應為 13）：", compute_brute_sum(test_grid, 1, 5, 0, 0, 3))

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.3：單點菱形和與平均值計算
# 任務說明：給定二維網格 grid 與指定中心 (cr, cc)，
# 請使用全圖暴力法計算該中心點對應數值 x 的菱形元素總和，
# 以及菱形內實際涵蓋的格子總數。回傳 (總和, 格子數)。
#
# 【公開測試資料 1】
# grid = [[3, 1, 4, 5, 1]], center = (0, 2) (中心值 x = 4)
# 預期輸出：(14, 5)  (覆蓋全部 5 格，總和 3+1+4+5+1 = 14)
#
# 【公開測試資料 2】
# grid = [[1, 2], [3, 4]], center = (0, 0) (中心值 x = 1)
# 預期輸出：(6, 3)   (覆蓋 (0,0)=1, (0,1)=2, (1,0)=3，總和 6)
# ==========================================
def diamond_sum_and_count(grid, cr, cc):
    # 請在下方撰寫你的程式碼：
    n = len(grid)
    m = len(grid[0])
    x = grid[cr][cc]
    total_sum = 0
    count = 0
    for r in range(n):
        for c in range(m):
            if abs(r - cr) + abs(c - cc) <= x:
                total_sum += grid[r][c]
                count += 1
    return (total_sum, count)

# 執行測試驗證
print("測試 1 結果：", diamond_sum_and_count([[3, 1, 4, 5, 1]], 0, 2))
print("測試 2 結果：", diamond_sum_and_count([[1, 2], [3, 4]], 0, 0))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.3：回傳所有貢獻格子的座標與數值
# 任務說明：實作函式 get_contributing_cells(grid, cr, cc)，
# 除了計算總和，還要回傳所有落在菱形範圍內的 [(r, c, val), ...] 清單。
# （本題為自由挑戰題，無公開測資，請自行設計並執行檢驗）
# ==========================================
def get_contributing_cells(grid, cr, cc):
    # 請在此處撰寫你的程式碼：
    n = len(grid)
    m = len(grid[0])
    x = grid[cr][cc]
    cells = []
    for r in range(n):
        for c in range(m):
            if abs(r - cr) + abs(c - cc) <= x:
                cells.append((r, c, grid[r][c]))
    return cells

sample = [[1, 3, 4], [1, 1, 4], [1, 1, 3]]
print("中心 (1, 1) 的貢獻格子清單：", get_contributing_cells(sample, 1, 1))

## 15.11.4 菱形區域求和演算法（二）：行範圍縮小法（列差 $dr$ 決定行範圍）與邊界防護

### 🚀 演算法躍升：從全圖掃描到幾何投影剪枝
雖然暴力法能通過小測資，但在演算法競賽的世界中，追求極致效能是卓越工程師與競賽選手的靈魂本能。
仔細觀察曼哈頓不等式：
$$|i - s| + |j - t| \le x$$

令列座標差為 $dr = s - i$（列偏移量）。顯然，只有當 $|dr| \le x$ 時，才有可能找到滿足條件的格子。換言之，$dr$ 的有效範圍必定在 $[-x, x]$ 之間！
當列偏移量 $dr$ 確定之後，我們可以把不等式移項：
$$|j - t| \le x - |dr|$$

這是一個驚人的數學化簡！
這意味著：**當列位移 $dr$ 固定時，行位移 $dc = t - j$ 的絕對值上限就只剩下 $rem = x - |dr|$**！
因此，該列在菱形內部的行座標範圍必定嚴格落在：
$$[j - rem, \quad j + rem]$$

```text
假設中心 (i, j)，半徑 x = 2：
dr = -2 (r = i-2) -> rem = 2 - |-2| = 0 -> 行範圍：[j, j]         (只有 1 格)
dr = -1 (r = i-1) -> rem = 2 - |-1| = 1 -> 行範圍：[j-1, j+1]     (共有 3 格)
dr =  0 (r = i)   -> rem = 2 - |0|  = 2 -> 行範圍：[j-2, j+2]     (共有 5 格)
dr = +1 (r = i+1) -> rem = 2 - |+1| = 1 -> 行範圍：[j-1, j+1]     (共有 3 格)
dr = +2 (r = i+2) -> rem = 2 - |+2| = 0 -> 行範圍：[j, j]         (只有 1 格)
```

### 🛡️ 邊界防禦機制：數值鉗制（Clamping）
當菱形靠近地圖邊緣時，算出的行座標可能小於 0 或大於等於 $m$；列座標也可能小於 0 或大於等於 $n$。
為了防止 `IndexError`，我們實施**邊界防禦**：
1. **列邊界檢查**：先確認 $0 \le i + dr < n$，若出界則直接 `continue` 跳過該列。
2. **行邊界鉗制**：
   * 起始行：$c_{start} = \max(0, j - rem)$
   * 結束行：$c_{end} = \min(m - 1, j + rem)$
   * 走訪範圍即為 `range(c_start, c_end + 1)`！

### ⚡ 效能震撼對比
題目給定陣列中的數值小於 10（$x \le 9$）。
半徑 $x \le 9$ 的菱形區域最多包含幾格？
$$	ext{最大格數} = 1 + 4 	imes rac{9 	imes 10}{2} = 1 + 180 = 181 	ext{ 格}$$
* 暴力法：每個中心檢查 $2500$ 格。
* 行範圍縮小法：每個中心**最多只檢查 181 格**！
運算量從 625 萬次驟降至約 45 萬次，**提速超過 13 倍**，執行時間縮短至 0.03 秒以內！

讓我們實作這套優雅且極速的高效求和函式。

In [ ]:
# 15.11.4 範例展示：行範圍縮小法（列差投影）高效菱形求和
def get_diamond_sum_optimized(grid, n, m, i, j, x):
    """
    行範圍縮小法：時間複雜度 O(x^2)，最多僅遍歷 181 格
    """
    diamond_sum = 0
    
    # 遍歷列位移量 dr ∈ [-x, x]
    for dr in range(-x, x + 1):
        r = i + dr
        # 邊界防禦 1：確認列座標在合法網格內
        if 0 <= r < n:
            # 計算當前列可分配的行位移預算
            rem = x - abs(dr)
            # 邊界防禦 2：行座標鉗制在 [0, m-1] 區間內
            c_start = max(0, j - rem)
            c_end = min(m - 1, j + rem)
            
            # 直接累加該列合法區間的數值
            for c in range(c_start, c_end + 1):
                diamond_sum += grid[r][c]
                
    return diamond_sum

# 驗證官方範例二中心 (2, 3), x = 2
sample_grid = [
    [1, 3, 4, 1, 3, 1],
    [1, 1, 4, 1, 3, 1],
    [1, 1, 3, 2, 5, 3],
    [4, 3, 3, 1, 4, 1],
    [5, 2, 1, 1, 1, 1]
]
n, m = len(sample_grid), len(sample_grid[0])
res = get_diamond_sum_optimized(sample_grid, n, m, 2, 3, 2)
print("優化演算法求和結果：", res)

In [ ]:
# 填空 15.11.4：補齊行範圍縮小法之列偏移與邊界鉗制
# 請將 ___ 替換為適當的運算式或變數

def optimized_diamond_sum(grid, n, m, cr, cc, x):
    total = 0
    for dr in range(-x, x + 1):
        r = cr + dr
        if 0 <= r < n:
            # 剩餘可分配給行差的額度
            rem = x - abs(___)
            # 使用 max 與 min 確保行索引不出界
            c_start = max(0, cc - ___)
            c_end = min(m - 1, cc + ___)
            for c in range(c_start, c_end + 1):
                total += grid[r][c]
    return total

print("測試中心 (2, 3) 結果（應為 32）：", optimized_diamond_sum(sample_grid, n, m, 2, 3, 2))

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.4：菱形逐列掃描區間檢視器
# 任務說明：給定網格尺寸 (n, m)、中心 (cr, cc) 與半徑 x。
# 請利用行範圍縮小法，回傳一個清單，記錄每一個合法列及其行掃描區間：
# 格式為 [(列, 起始行, 結束行, 涵蓋格數), ...]
#
# 【公開測試資料 1】
# n = 5, m = 5, 中心 = (2, 2), x = 1
# 預期輸出：[(1, 2, 2, 1), (2, 1, 3, 3), (3, 2, 2, 1)]
#
# 【公開測試資料 2】
# n = 3, m = 3, 中心 = (0, 0), x = 1
# 預期輸出：[(0, 0, 1, 2), (1, 0, 0, 1)]
# ==========================================
def inspect_diamond_intervals(n, m, cr, cc, x):
    # 請在下方撰寫你的程式碼：
    intervals = []
    for dr in range(-x, x + 1):
        r = cr + dr
        if 0 <= r < n:
            rem = x - abs(dr)
            c_start = max(0, cc - rem)
            c_end = min(m - 1, cc + rem)
            if c_start <= c_end:
                count = c_end - c_start + 1
                intervals.append((r, c_start, c_end, count))
    return intervals

# 執行測試驗證
print("測試 1 結果：", inspect_diamond_intervals(5, 5, 2, 2, 1))
print("測試 2 結果：", inspect_diamond_intervals(3, 3, 0, 0, 1))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.4：暴力法 vs 優化法結果一致性對拍驗證
# 任務說明：隨機生成一個 10 x 10 的測試網格（數值在 1~9 之間），
# 對網格中的每一個格子，分別以「暴力法」與「行縮小優化法」計算菱形總和。
# 驗證兩者計算出來的總和是否 100% 完全相同。
# （本題為自由挑戰題，無公開測資，請自行設計程式碼）
# ==========================================
import random

def verify_algorithms_consistency():
    # 請在此處撰寫你的程式碼：
    random.seed(42)
    n, m = 10, 10
    test_grid = [[random.randint(1, 9) for _ in range(m)] for _ in range(n)]
    
    all_matched = True
    for i in range(n):
        for j in range(m):
            x = test_grid[i][j]
            s_brute = get_diamond_sum_brute(test_grid, n, m, i, j, x)
            s_opt = get_diamond_sum_optimized(test_grid, n, m, i, j, x)
            if s_brute != s_opt:
                all_matched = False
                break
    return all_matched

print("兩種演算法 100 個點對拍驗證結果是否完全吻合：", verify_algorithms_consistency())

## 15.11.5 特殊位置判定條件：元素和模 10 與中心值模 10 之餘數比對（$sum \pmod{10} == x \pmod{10}$）

### 🎯 核心判準拆解：何謂「特殊位置」？
在完成幾何求和模組後，我們來到了題目的靈魂核心判定準則。
官方題面明確給出定義：
> 「對於一個元素 $A[i][j] = x$，若與其距離不超過 $x$ 的元素總和（包含 $A[i][j]$ 本身）除以 10 的餘數，與 $x$ 除以 10 的餘數相同，則稱 $(i, j)$ 為一特殊位置。」

在 Python 語言中，「除以 10 的餘數」使用取模運算子 `%`（Modulus Operator）：
```python
is_special = (diamond_sum % 10 == x % 10)
```

### 🧠 深入探索：題目條件下的數學特徵
1. **元素數值皆小於 10**：
   輸入規範保證「陣列內的數字均為小於 10 的正整數」，即 $x \in \{1, 2, 3, 4, 5, 6, 7, 8, 9\}$。
   因此，對於中心值 $x$ 而言，$x \pmod{10}$ 的值其實恆等於 $x$ 本身！
   例如：$2 \pmod{10} = 2$、$9 \pmod{10} = 9$。
2. **防禦性程式設計（Defensive Programming）**：
   儘管 $x < 10$，在撰寫代碼時，依然強烈建議寫成 `diamond_sum % 10 == x % 10`。
   這樣做有兩大好處：
   * 忠實反映題目的語意要求，代碼自帶解說文檔。
   * 若未來題目升級改版，出現 $x \ge 10$ 的測資，這行代碼依然能 100% 正確無誤！

### ⚠️ 運算子優先級與考場常見筆誤
* **優先級**：在 Python 運算子優先順序中，算術運算子 `%` 的優先級高於比較運算子 `==`。
  因此 `diamond_sum % 10 == x % 10` 會先計算兩側的餘數，再進行相等判定，語法完全安全。
* **混淆 `%` 與 `//`**：千萬不要誤打成整數除法 `//`（商數）。
* **非嚴格等於**：切勿漏掉 `=` 寫成賦值符號。

讓我們封裝一個獨立的特殊位置布林檢驗函式。

In [ ]:
# 15.11.5 範例展示：特殊位置布林判定函式
def is_special_position(grid, n, m, i, j):
    """
    判定座標 (i, j) 是否為特殊位置
    回傳：(is_special, x, diamond_sum, remainder)
    """
    x = grid[i][j]
    diamond_sum = get_diamond_sum_optimized(grid, n, m, i, j, x)
    
    # 核心條件：菱形總和 mod 10 是否等於 x mod 10
    is_special = (diamond_sum % 10 == x % 10)
    
    return is_special, x, diamond_sum, diamond_sum % 10

# 測試官方範例二的幾個關鍵座標
test_coords = [(0, 0), (0, 2), (2, 3)]
for r, c in test_coords:
    res, val, s_sum, rem = is_special_position(sample_grid, n, m, r, c)
    status_str = "⭐ 是特殊位置" if res else "❌ 不是特殊位置"
    print(f"座標 ({r}, {c}) | 中心值={val} | 菱形和={s_sum:2d} | 餘數={rem} | {status_str}")

In [ ]:
# 填空 15.11.5：補齊模 10 餘數比對與布林判定條件
# 請將 ___ 替換為適當的運算式或變數

def verify_special_rule(diamond_sum, center_value):
    # 比較菱形總和除以 10 的餘數 與 中心值除以 10 的餘數
    remainder_sum = diamond_sum % ___
    remainder_val = center_value % ___
    return remainder_sum == ___

print("總和 32, 中心值 2 是否匹配：", verify_special_rule(32, 2))
print("總和 15, 中心值 1 是否匹配：", verify_special_rule(15, 1))

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.5：網格全體特殊位置快篩器
# 任務說明：給定二維網格 grid。
# 請走訪整個網格，呼叫判定函式，回傳所有符合條件的特殊位置三元組：
# 格式為 [(列標, 行標, 中心值), ...]
#
# 【公開測試資料 1】
# grid = [[3, 1, 4, 5, 1]] (官方範例一)
# 預期輸出：[(0, 0, 3), (0, 2, 4)]
#
# 【公開測試資料 2】
# grid = [[2, 5]] (邊界測資：無符合者)
# 預期輸出：[]
# ==========================================
def scan_special_cells(grid):
    # 請在下方撰寫你的程式碼：
    n = len(grid)
    m = len(grid[0])
    specials = []
    for r in range(n):
        for c in range(m):
            x = grid[r][c]
            s = get_diamond_sum_optimized(grid, n, m, r, c, x)
            if s % 10 == x % 10:
                specials.append((r, c, x))
    return specials

# 執行測試驗證
print("測試 1 結果：", scan_special_cells([[3, 1, 4, 5, 1]]))
print("測試 2 結果：", scan_special_cells([[2, 5]]))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.5：餘數基底參數化擴充
# 任務說明：若教育部競賽委員會將規則改進為：
# 「總和除以基底 K 的餘數等於中心值除以基底 K 的餘數」。
# 請撰寫函式 find_special_base_k(grid, k)，支援任意基底 K（例如 K = 7 或 K = 16）。
# （本題為自由挑戰題，無公開測資，請自行設計程式碼）
# ==========================================
def find_special_base_k(grid, k):
    # 請在此處撰寫你的程式碼：
    n = len(grid)
    m = len(grid[0])
    results = []
    for r in range(n):
        for c in range(m):
            x = grid[r][c]
            s = get_diamond_sum_optimized(grid, n, m, r, c, x)
            if s % k == x % k:
                results.append((r, c))
    return results

print("基底 K=5 時的特殊位置：", find_special_base_k([[3, 1, 4, 5, 1]], 5))

## 15.11.6 多解座標收集與自然字典序排序（先比列標 $i$ 再比行標 $j$）

### 🎯 題目輸出規格深度解碼
在 APCS 評測系統中，答案正確但「格式不符」或「順序錯誤」將導致慘痛的 `WA`（Wrong Answer）。
讓我們再次複習官方題本中的輸出規範：
1. **第一行輸出一個整數 $k$**：表示所求之特殊位置總數。
2. **接下來 $k$ 行**：每行輸出兩個以單一空格隔開的整數，分別代表該特殊位置的列註標與行註標。
3. **排序要求**：**「列註標較小的先輸出；若列註標相同，則先輸出行註標較小者」**。

### ✨ Python 自然走訪的「天賜禮物」：天然字典序！
初學者看到「排序要求」，常直覺地想要調用 `.sort()` 或自訂排序比較函式。
但在本題中，請觀察我們的走訪架構：
```python
for i in range(n):        # i 從 0 遞增到 n-1
    for j in range(m):    # 相同 i 下，j 從 0 遞增到 m-1
        if is_special(i, j):
            results.append((i, j))
```
* 第一層迴圈由上而下掃描列，所以 $i$ 必定是由小到大！
* 第二層迴圈在同一列內由左至右掃描行，所以當 $i$ 相同時，$j$ 也必定是由小到大！
這意味著：**只要我們依照標準的「先列後行」雙重迴圈逐一檢查並 `append`，清單中的座標本身就已經嚴格保證符合題目要求的字典序**！完全不需要進行任何額外排序！

### 🛡️ 雙重防護：呼叫 `results.sort()` 的零成本保險
儘管自然走訪已有序，在考場上若擔心自己中途被打亂，直接加上 `results.sort()` 也是絕佳的好習慣。
因為在 Python 中，對元組串列 `[(r, c), ...]` 呼叫 `.sort()` 時，預設的比對規則正是**「先比第 0 個元素，若相同再比第 1 個元素」**，與題意 100% 完美契合！

### ⚠️ 無解特例防呆（$k = 0$）
若整張地圖沒有任何特殊位置：
* 第一行必須輸出 `0`。
* 接下來**不可輸出任何多餘的空行或空白**，否則會被評判系統視為格式錯誤（PE/WA）。

讓我們透過程式碼演練完整的收集與格式化輸出流程。

In [ ]:
# 15.11.6 範例展示：多解座標收集與符合官方規範的輸出格式化
def format_apcs_output(special_list):
    """
    依官方規範輸出特殊位置結果：
    第 1 行輸出數量 k；接下來 k 行輸出列 行 座標
    """
    k = len(special_list)
    print(k)
    for r, c in special_list:
        print(f"{r} {c}")

# 模擬收集到的座標清單（故意打亂順序驗證排序）
disordered_coords = [(2, 3), (0, 2), (3, 3), (1, 3)]
print("未排序前：", disordered_coords)

# 執行 Python 內建元組排序
disordered_coords.sort()
print("字典序排序後：", disordered_coords)
print("-" * 30)
print("【標準 APCS 終端輸出呈現】")
format_apcs_output(disordered_coords)

In [ ]:
# 填空 15.11.6：補齊座標收集與標準格式化輸出
# 請將 ___ 替換為適當的運算式或變數

def collect_and_print_specials(specials):
    # 防禦性字典序排序
    specials.___()
    
    # 步驟 1：輸出符合條件的總數量
    print(len(___))
    
    # 步驟 2：逐行輸出列標與行標
    for r, c in specials:
        print(f"{r} {___}")

test_data = [(0, 0), (0, 2)]
collect_and_print_specials(test_data)

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.6：無序座標重整與格式化輸出器
# 任務說明：接收一組混亂且可能重複的座標串列 raw_coords。
# 請先去除重複座標，接著依列標小到大、行標小到大排序，
# 並回傳格式化字串（首行為數量，後續為各座標，以換行分隔）。
#
# 【公開測試資料 1】
# raw_coords = [(1, 3), (0, 2), (1, 3), (2, 3)]
# 預期輸出字串：
# 3
# 0 2
# 1 3
# 2 3
#
# 【公開測試資料 2】
# raw_coords = []
# 預期輸出字串：
# 0
# ==========================================
def format_unique_sorted_coords(raw_coords):
    # 請在下方撰寫你的程式碼：
    unique_sorted = sorted(list(set(raw_coords)))
    lines = [str(len(unique_sorted))]
    for r, c in unique_sorted:
        lines.append(f"{r} {c}")
    return "\n".join(lines)

# 執行測試驗證
print("測試 1 結果：\n" + format_unique_sorted_coords([(1, 3), (0, 2), (1, 3), (2, 3)]))
print("-" * 20)
print("測試 2 結果：\n" + format_unique_sorted_coords([]))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.6：逆向字典序輸出格式轉換
# 任務說明：若延伸題目要求「列標由大到小輸出；列標相同時，行標亦由大到小輸出」。
# 請撰寫函式 format_reverse_lexicographical(coords) 實現此功能。
# （本題為自由挑戰題，無公開測資，請自行設計程式碼）
# ==========================================
def format_reverse_lexicographical(coords):
    # 請在此處撰寫你的程式碼：
    sorted_coords = sorted(coords, key=lambda p: (p[0], p[1]), reverse=True)
    out = [str(len(sorted_coords))]
    for r, c in sorted_coords:
        out.append(f"{r} {c}")
    return "\n".join(out)

sample_pts = [(0, 2), (1, 3), (2, 3), (3, 3)]
print("逆向字典序輸出：\n" + format_reverse_lexicographical(sample_pts))

## 15.11.7 完整 AC 模組組裝：雙子題組（$n=1$ 一維特例與 $50 	imes 50$ 二維）通關程式

### 🎯 雙子題組架構融合：一維與二維的統一之美
在審視題目評分說明時，我們看到了：
* **子題組 1（60分）**：$n = 1$（網格只有一列）。
* **子題組 2（40分）**：無額外限制（完整二維矩陣，$1 \le n, m \le 50$）。

許多初學者在考場上看到子題組 1，會想額外寫一個 `if n == 1:` 的分支去單獨處理一維陣列。
然而在優秀的程式架構中，**一維陣列本質上就是只有 1 列（$n=1$）的二維陣列**！
當 $n = 1$ 時：
* 外層列迴圈只會執行一次（$i = 0$）。
* 在行範圍縮小法中，只有 $dr = 0$ 能通過 $0 \le 0 + dr < 1$ 的檢查。
* 程式會自動簡化為在一維線上左右延伸 $rem = x$ 的範圍進行求和！
因此，**一套統一且優雅的二維曼哈頓演算法，能同時完美擊穿兩大子題組，直接斬獲 100 分滿分**！

### 🧩 完整通關組裝 5 部曲
1. **輸入解析**：讀取第一行取得 $n, m$，接續讀取 $n$ 行整數陣列建構 `grid`。
2. **初始化容器**：建立 `specials = []` 存放特殊位置。
3. **主走訪引擎**：雙重迴圈遍歷每個格子 $(i, j)$，取得中心值 $x = grid[i][j]$。
4. **高效求和與判定**：以列位移 $dr \in [-x, x]$ 掃描，加總有效格子並檢查 `sum % 10 == x % 10`。
5. **規範輸出**：依序印出數量與各座標。

讓我們將所有模組組裝為一套完整的、可以直接在考場提交的單元測試程式。

In [ ]:
# 15.11.7 範例展示：完整 AC 解題模組組裝與官方範例一、二全通過驗證
def solve_special_positions(n, m, grid):
    """
    整合模組：計算 n x m 網格的所有特殊位置
    """
    specials = []
    
    # 步驟 1：逐格走訪中心點 (i, j)
    for i in range(n):
        for j in range(m):
            x = grid[i][j]
            diamond_sum = 0
            
            # 步驟 2：高效行範圍縮小法求和
            for dr in range(-x, x + 1):
                r = i + dr
                if 0 <= r < n:
                    rem = x - abs(dr)
                    c_start = max(0, j - rem)
                    c_end = min(m - 1, j + rem)
                    for c in range(c_start, c_end + 1):
                        diamond_sum += grid[r][c]
            
            # 步驟 3：模 10 餘數比對判定
            if diamond_sum % 10 == x % 10:
                specials.append((i, j))
                
    return specials

# 驗證官方範例一 (子題 1: n = 1)
ans1 = solve_special_positions(1, 5, [[3, 1, 4, 5, 1]])
print("官方範例一（一維子題）解題結果：", ans1)

# 驗證官方範例二 (子題 2: 二維)
ans2 = solve_special_positions(5, 6, [
    [1, 3, 4, 1, 3, 1],
    [1, 1, 4, 1, 3, 1],
    [1, 1, 3, 2, 5, 3],
    [4, 3, 3, 1, 4, 1],
    [5, 2, 1, 1, 1, 1]
])
print("官方範例二（二維完整）解題結果：", ans2)

In [ ]:
# 填空 15.11.7：補齊完整求解骨架之核心迴圈與條件
# 請將 ___ 替換為適當的運算式或變數

def complete_solver_skeleton(n, m, grid):
    res = []
    for r in range(n):
        for c in range(m):
            val = grid[r][c]
            s = 0
            for dr in range(-val, val + 1):
                nr = r + dr
                if 0 <= nr < n:
                    rem = val - abs(dr)
                    # 累加行範圍
                    for nc in range(max(0, c - rem), min(m - 1, c + rem) + 1):
                        s += grid[nr][nc]
            # 判定餘數
            if s % 10 == val % ___:
                res.append((r, ___))
    return res

print("填空檢驗結果：", complete_solver_skeleton(1, 5, [[3, 1, 4, 5, 1]]))

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.7：字串輸入流模組化解析與解答
# 任務說明：接收模擬考場終端輸入的純文字字串 input_str，
# 依序解析出 n, m 與 grid，並回傳格式化的最終輸出字串。
#
# 【公開測試資料 1】
# input_str = """1 5
# 3 1 4 5 1"""
# 預期輸出字串：
# 2
# 0 0
# 0 2
#
# 【公開測試資料 2】
# input_str = """3 3
# 1 2 1
# 2 3 2
# 1 2 1"""
# 預期輸出字串：
# 1
# 1 1
# ==========================================
def solve_from_input_string(input_str):
    # 請在下方撰寫你的程式碼：
    tokens = input_str.strip().split()
    if not tokens:
        return "0"
    n = int(tokens[0])
    m = int(tokens[1])
    idx = 2
    grid = []
    for _ in range(n):
        grid.append([int(tokens[idx + c]) for c in range(m)])
        idx += m
    
    specials = solve_special_positions(n, m, grid)
    out = [str(len(specials))]
    for r, c in specials:
        out.append(f"{r} {c}")
    return "\n".join(out)

# 執行測試驗證
tc1 = """1 5
3 1 4 5 1"""
tc2 = """3 3
1 2 1
2 3 2
1 2 1"""
print("測試 1 結果：\n" + solve_from_input_string(tc1))
print("-" * 20)
print("測試 2 結果：\n" + solve_from_input_string(tc2))

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.7：多矩陣批量自動化測試器
# 任務說明：撰寫函式 batch_benchmark_runner(case_list)，
# 接收多個測資字典 [{"n": ..., "m": ..., "grid": ..., "expected": ...}]，
# 自動逐一執行並印出 PASS / FAIL 報告。
# （本題為自由挑戰題，無公開測資，請自行設計程式碼）
# ==========================================
def batch_benchmark_runner(case_list):
    # 請在此處撰寫你的程式碼：
    all_ok = True
    for i, tc in enumerate(case_list, 1):
        actual = solve_special_positions(tc["n"], tc["m"], tc["grid"])
        status = "PASS" if actual == tc["expected"] else "FAIL"
        if status == "FAIL":
            all_ok = False
        print(f"Case {i}: [{status}] Actual={actual}, Expected={tc['expected']}")
    return all_ok

demo_cases = [
    {"n": 1, "m": 5, "grid": [[3, 1, 4, 5, 1]], "expected": [(0, 0), (0, 2)]},
    {"n": 1, "m": 1, "grid": [[3]], "expected": [(0, 0)]}
]
print("批次測試全數通過：", batch_benchmark_runner(demo_cases))

## 15.11.8 極端邊界壓力測試（單點、邊緣菱形溢出、無符合位置）、複雜度分析與考場 WA 排查

### 🔍 演算法時空複雜度權威剖析
在正式進入考場之前，卓越的選手必須能夠在腦海中精確量化代碼的資源開銷：

1. **時間複雜度（Time Complexity）**：
   * **走訪中心點次數**：$n 	imes m$ 次，最大為 $50 	imes 50 = 2500$ 次。
   * **單一中心點求和次數**：列偏移 $dr \in [-x, x]$，因為陣列元素皆小於 10（$x \le 9$），所以最多走訪 19 列。每列走訪格數最多為 $2(x - |dr|) + 1$。
   * **菱形最大總格數**：
     $$S_{max} = 1 + 2 	imes 9 	imes 10 = 181 	ext{ 格}$$
   * **全局總運算次數上限**：
     $$T_{max} \le 2500 	imes 181 pprox 4.52 	imes 10^5 	ext{ 次加法}$$
   * **結論**：在現代 Python 直譯環境下，45 萬次基本整數運算耗時約 **0.02～0.04 秒**，遠低於官方 1.0 秒的時間限制，以極充裕的效能餘裕穩穩 AC！

2. **空間複雜度（Space Complexity）**：
   * 儲存 $n 	imes m$ 矩陣：$50 	imes 50$ 個整數，記憶體佔用小於 100 KB。
   * 答案串列 `specials`：最多存放 2500 個元組。
   * **全局輔助空間**：$O(n 	imes m)$，記憶體消耗小於 1 MB，遠低於官方 256 MB 上限。

---

### ⚠️ 考場常見致命 WA 地雷 Checklist
在考場緊張的環境下，即使思路正確，也常因細節疏忽而痛失級分。請牢記以下排查清單：

| 地雷類型 | 致命錯誤寫法 | 正確防禦解法 |
| :--- | :--- | :--- |
| **地雷一：菱形擴散越界** | `grid[i + dr][j + dc]` 未加保護 | 必須檢查 $0 \le i + dr < n$ 且利用 `max(0, ...)`、`min(m - 1, ...)` 嚴格限制行索引 |
| **地雷二：遺漏輸出數量** | 直接輸出座標，忘了第 1 行的總數 $k$ | 第一行務必先輸出 `print(len(specials))` |
| **地雷三：無解時格式錯誤** | $k=0$ 時印出空行或崩潰 | 當清單為空時，第一行輸出 `0`，且後續不輸出任何座標 |
| **地雷四：排序維度倒置** | 先排行再排列（$c$ 優先於 $r$） | 嚴格遵守「先列後行」，直接使用自然走訪或 `sort()` 即可 |
| **地雷五：模運算基底記錯** | 誤寫為 `% x` 或 `% 9` | 題意明確要求「除以 10 的餘數」，必須嚴格 `% 10` |

讓我們實作四大極端壓力測資自動檢驗套件。

In [ ]:
# 15.11.8 範例展示：四大極端壓力與邊界測資自動回歸檢驗
def run_extreme_boundary_tests():
    test_suite = [
        {
            "name": "極端測資 1：最小單點矩陣 (1x1, A[0][0] = 3)",
            "n": 1, "m": 1,
            "grid": [[3]],
            "expected_count": 1,
            "expected_coords": [(0, 0)]
        },
        {
            "name": "極端測資 2：邊角極限溢出 (2x2 全為 9，菱形大幅溢出地圖外)",
            "n": 2, "m": 2,
            "grid": [[9, 9], [9, 9]],
            "expected_count": 0,  # 總和 36 % 10 = 6 != 9，無特殊位置
            "expected_coords": []
        },
        {
            "name": "極端測資 3：無任何符合位置 (1x2, [2, 5])",
            "n": 1, "m": 2,
            "grid": [[2, 5]],
            "expected_count": 0,
            "expected_coords": []
        },
        {
            "name": "極端測資 4：全數皆為特殊位置 (2x2 元素為 5，總和 20 % 10 = 0 != 5；構造符合例)",
            "n": 1, "m": 3,
            "grid": [[2, 4, 2]],  # 檢查 (0,1)=4: 總和 8 != 4; (0,0)=2: 總和 2+4=6!=2
            "expected_count": 0,
            "expected_coords": []
        }
    ]
    
    print("=" * 65)
    print("🛡️ 開始執行極端邊界測試...")
    print("=" * 65)
    for tc in test_suite:
        actual = solve_special_positions(tc["n"], tc["m"], tc["grid"])
        passed = (actual == tc["expected_coords"]) and (len(actual) == tc["expected_count"])
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"[{status}] {tc['name']}")
        print(f"       實際得到數量: {len(actual)}，座標: {actual}")
    print("=" * 65)

run_extreme_boundary_tests()

In [ ]:
# 填空 15.11.8：計算理論最大運算次數與時間估算
# 請將 ___ 替換為適當的常數或運算式

def calculate_theoretical_max_operations(max_n, max_m, max_val):
    # 網格最多格子數
    max_cells = max_n * max_m
    # 半徑為 max_val 時的菱形最大格數 (1 + 2 * x * (x + 1))
    max_diamond_cells = 1 + 2 * max_val * (max_val + ___)
    # 理論最大走訪加總次數
    total_ops = max_cells * ___
    return total_ops

# 帶入 APCS 規格：n=50, m=50, max_val=9
ops = calculate_theoretical_max_operations(50, 50, 9)
print(f"APCS 測資最大理論運算次數：{ops} 次（約 {ops / 10000:.1f} 萬次）")

In [ ]:
# ==========================================
# [4] Code 練習題 15.11.8：滿載 50x50 隨機矩陣極限效能壓力測試
# 任務說明：隨機生成 50x50 的最大規格矩陣，元素值在 1~9 之間。
# 測量 solve_special_positions 在滿載測資下的執行秒數，
# 驗證執行時間必須嚴格小於 0.2 秒（遠低於 1.0 秒限制）。
#
# 【公開測試資料 1】
# 規格：50 x 50 隨機網格
# 預期輸出：執行秒數 < 0.2 秒，狀態為 PASS
#
# 【公開測試資料 2】
# 規格：1 x 50 最大一維子題網格
# 預期輸出：執行秒數 < 0.05 秒，狀態為 PASS
# ==========================================
import time
import random

def run_performance_stress_test():
    random.seed(202306)
    # 測試 1：50x50 滿載二維矩陣
    grid_50x50 = [[random.randint(1, 9) for _ in range(50)] for _ in range(50)]
    t0 = time.perf_counter()
    res1 = solve_special_positions(50, 50, grid_50x50)
    t1 = time.perf_counter()
    elapsed1 = t1 - t0
    
    # 測試 2：1x50 一維子題
    grid_1x50 = [[random.randint(1, 9) for _ in range(50)]]
    t2 = time.perf_counter()
    res2 = solve_special_positions(1, 50, grid_1x50)
    t3 = time.perf_counter()
    elapsed2 = t3 - t2
    
    print(f"50x50 耗時：{elapsed1:.4f} 秒 -> {'✅ PASS (< 0.2s)' if elapsed1 < 0.2 else '❌ TIME LIMIT'}")
    print(f"1x50  耗時：{elapsed2:.4f} 秒 -> {'✅ PASS (< 0.05s)' if elapsed2 < 0.05 else '❌ TIME LIMIT'}")
    return elapsed1 < 0.2 and elapsed2 < 0.05

run_performance_stress_test()

In [ ]:
# ==========================================
# [5] Code 挑戰題 15.11.8：特殊位置反向建構器
# 任務說明：設計一個小工具 generate_guaranteed_special_grid(n, m)，
# 嘗試構建一個小型網格，使得 (0, 0) 必然是特殊位置。
# （本題為自由挑戰題，無公開測資，請自行設計程式碼驗證）
# ==========================================
def generate_guaranteed_special_grid():
    # 請在此處撰寫你的程式碼：
    # 構建一個 1x1 網格，值為任意 1~9，必然是特殊位置
    return [[7]]

sample_created = generate_guaranteed_special_grid()
print("反向構建網格：", sample_created)
print("驗證是否為特殊位置：", solve_special_positions(len(sample_created), len(sample_created[0]), sample_created))

## 🏆 恭喜通關！單元學習總結與解鎖能力盤點

### 🌟 今日解鎖核心能力盤點
恭喜各位程式冒險者！透過本單元 8 大漸進式學習階梯的扎實特訓，你已經成功攻克了教育部官方中級題本的第一道門戶大題——**k732. 特殊位置**！
讓我們盤點你在本單元中解鎖的高階競技能力：
- [x] **曼哈頓距離幾何模型**：深刻理解網格步數距離與 45 度旋轉菱形覆蓋區域的數學本質。
- [x] **二維陣列安全走訪功底**：精通「先列後行」的存取規範，剷除 `IndexError` 與維度混淆的失分死穴。
- [x] **行範圍縮小法（投影剪枝）**：推導 $|dr| + |dc| \le x \implies |dc| \le x - |dr|$，將單格走訪運算量從 $O(N \cdot M)$ 暴降至最多 181 次，效能飛躍十倍以上！
- [x] **邊界鉗制防護技術**：精熟運用 `max(0, ...)` 與 `min(m - 1, ...)` 進行安全夾止，從容抵禦菱形溢出邊界。
- [x] **天然字典序輸出機制**：領會 Python 外層列、內層行循序走訪自帶的有序特性，以零成本滿足嚴格的官方排序要求。
- [x] **一維與二維統一架構**：領悟「一維即單列二維」的降維本質，一套代碼同時通吃 60 分子題與 40 分子題，穩穩斬獲 100 分滿分 AC！

```text
    ╭────────────────────────────────────────────────────────╮
    │  🏅 恭喜獲得榮耀通關徽章：【曼哈頓幾何掃描大師】      │
    │  解鎖試題：k732. 特殊位置（APCS 2023-06 實作第 2 題）   │
    │  下一站挑戰：15.12 o712. 蒐集寶石（方向向量與動態模擬） │
    ╰────────────────────────────────────────────────────────╯
```

## 💻 【附錄：雙平台滿分通關解答庫】考 APCS vs 刷 ZeroJudge 對照

在線上刷題與實體考場之間，許多學習者常因「輸入評判機制不同」或「寫法過於精簡/冗長」而感到困惑。
為了讓所有初學同學與進階選手都能找到最適合自己的武器，本附錄特別提供**三大獨立滿分版本**：

| 評測與程式版本 | 適用情境與受眾推薦 | 核心寫法特色與優勢 |
| :--- | :--- | :--- |
| **🥇 版本一：APCS 淺顯易懂一般版** | APCS 正式考場（**初學同學首選推薦**） | 步驟平鋪直敘、標準 `for` 迴圈逐行讀取、變數語意直白明確。考場高壓下思路最清晰、最不易緊張出錯，穩健 100% AC！ |
| **⚡ 版本二：APCS 極簡高效精煉版** | APCS 正式考場（**進階競賽選手推薦**） | 運用現代 Pythonic 串列生成式與切片求和，代碼極度緊湊精煉，展現高階極致效能與優雅架構！ |
| **🌐 版本三：ZeroJudge 萬用 AC 版** | ZeroJudge 線上評判系統（題號：k732） | 針對線上評判連續多筆測資串流設計，支援 `sys.stdin` 迴圈至 EOF，並內建「Colab 本地自動化測試檢驗展示」！ |

### 📝 版本一：APCS 官方實作考場專用版 —— 淺顯易懂一般版（新手友善推薦）

* **適用情境**：APCS 正式考試現場（題目保證單筆測資輸入）。
* **教學與設計理念**：
  1. **平鋪直敘、步驟展開**：不堆疊資訊密度過高的一行寫法，而是使用最直觀清晰的標準 `for` 迴圈，將「讀取輸入 ➔ 逐格走訪 ➔ 行範圍縮小 ➔ 餘數判定 ➔ 字典序輸出」拆解為五個清晰步驟。
  2. **語意明確易除錯**：每個變數（如 `diamond_sum`、`rem_dist`、`col_start`、`col_end`）皆具備直觀命名，即便在考場緊張氛圍下，也能隨時隨地看懂每一步在做什麼。
  3. **穩健 100% 滿分 AC**：雖然行數約 35 行，但邏輯完全透明，零冷門黑魔法，保證穩穩奪下 100 分！

In [ ]:
# ==============================================================================
# 📝 版本一：APCS 官方實作考場專用版 —— 淺顯易懂一般版（新手友善推薦）
# 適用情境：APCS 正式考場單筆輸入保證，平鋪直敘、拆解詳細、穩健滿分
# ==============================================================================

def solve_apcs_version1():
    # 步驟 1：讀取矩陣的列數 n 與行數 m
    first_line = input().split()
    if not first_line:
        return
    n = int(first_line[0])
    m = int(first_line[1])

    # 步驟 2：逐列讀取網格資料，建立二維串列 grid
    grid = []
    for _ in range(n):
        row = [int(val) for val in input().split()]
        grid.append(row)

    # 步驟 3：建立特殊位置的收集清單
    special_positions = []

    # 步驟 4：雙重迴圈逐一走訪每個格子 (i, j) 作為中心點
    for i in range(n):
        for j in range(m):
            x = grid[i][j]  # 中心格子的數值，同時也是曼哈頓距離半徑
            diamond_sum = 0 # 累加菱形半徑內的格子數值

            # 步驟 5：以列偏移量 dr 在 [-x, x] 範圍內走訪
            for dr in range(-x, x + 1):
                r = i + dr  # 計算目標列座標
                # 檢查目標列是否在地圖邊界內
                if 0 <= r < n:
                    # 曼哈頓距離剩餘分配給行位移的預算
                    rem_dist = x - abs(dr)
                    # 計算合法行座標起訖點（利用 max 與 min 確保不越界）
                    col_start = max(0, j - rem_dist)
                    col_end = min(m - 1, j + rem_dist)
                    # 累加該列在菱形範圍內的所有格子
                    for c in range(col_start, col_end + 1):
                        diamond_sum += grid[r][c]

            # 步驟 6：判定總和除以 10 的餘數是否等於中心值除以 10 的餘數
            if diamond_sum % 10 == x % 10:
                special_positions.append((i, j))

    # 步驟 7：依題目規範輸出結果
    # 第一行輸出特殊位置總數
    print(len(special_positions))
    # 接下來每行依序輸出列註標與行註標（由小到大字典序輸出）
    for r, c in special_positions:
        print(f"{r} {c}")

if __name__ == "__main__":
    solve_apcs_version1()

### ⚡ 版本二：APCS 官方實作考場專用版 —— 極簡高效精煉版（進階高手推薦）

* **適用情境**：APCS 正式考試現場（追求極短程式碼、快速敲碼的高階模式）。
* **教學與設計理念**：
  1. **展現 Python 現代語法極致表現力**：透過 `sys.stdin.read().split()` 一口氣讀取所有輸入資料，並利用列表生成式（List Comprehension）直接建構網格。
  2. **高階切片求和（Slice Summing）**：在行範圍求和時，直接利用 Python 原生的高速切片 `grid[i + dr][c_start : c_end + 1]` 搭配 C 語言底層優化的 `sum()`，取代逐格 `for` 迴圈！
  3. **極度短小精悍**：全篇核心代碼僅約 15 行，敲碼極速，運算效能更是位居頂尖！

In [ ]:
# ==============================================================================
# ⚡ 版本二：APCS 官方實作考場專用版 —— 極簡高效精煉版（進階高手推薦）
# 適用情境：APCS 正式考場，展現 Python 極短程式碼之高階表現力與極致效能
# ==============================================================================

import sys

def solve_apcs_version2():
    # 一次性讀入所有單詞
    tokens = sys.stdin.read().split()
    if not tokens:
        return
    n, m = int(tokens[0]), int(tokens[1])
    grid = [list(map(int, tokens[2 + i * m : 2 + (i + 1) * m])) for i in range(n)]

    # 運用列表生成式與切片求和極速篩選特殊位置
    specials = [
        (i, j)
        for i in range(n)
        for j in range(m)
        if (
            sum(
                sum(grid[i + dr][max(0, j - (grid[i][j] - abs(dr))) : min(m, j + (grid[i][j] - abs(dr)) + 1)])
                for dr in range(-grid[i][j], grid[i][j] + 1)
                if 0 <= i + dr < n
            ) % 10 == grid[i][j] % 10
        )
    ]

    # 輸出數量與座標
    print(len(specials))
    for r, c in specials:
        print(f"{r} {c}")

if __name__ == "__main__":
    solve_apcs_version2()

### 🌐 版本三：ZeroJudge 線上評判萬用 AC 版（支援多筆測資 EOF 循環）

* **適用情境**：ZeroJudge 線上解題系統（題目代碼：`k732`）、各高中競賽線上評判平台。
* **解題特點與機制剖析**：
  1. **為什麼需要支援 EOF 多測資？**  
     在 ZeroJudge 等線上 OJ（Online Judge）系統中，評測主機會將整份測試檔（可能包含數十筆測資）一口氣管線重導向（Pipe）輸入程式中。若程式只讀一次就結束退出，評測系統將直接判定為 `WA`。
  2. **全自動 Tokens 迭代解析器**：  
     利用 `sys.stdin.read().split()` 將全檔所有字串單詞以空白與換行切開，利用索引指標 `idx` 一筆接著一筆讀取 $n, m$ 與二維矩陣，直到檔案結尾（EOF）。這種做法具有「免疫任意多餘空行、換行與空格」的強大穩健度！
  3. **內建【Colab 本地自動化測試檢驗展示】**：  
     下方 Code Cell 內嵌了官方範例一、官方範例二以及兩大極端邊界測資的自動化測試驅動器。學習者在 Colab 上只需點擊播放鍵，即可在本地瞬間印出全綠燈的測試通過報告！

In [ ]:
# ==============================================================================
# 🌐 版本三：ZeroJudge 線上評判萬用 AC 版（支援多筆測資 EOF 循環）
# 官方題號：ZeroJudge k732 / APCS 2023-06 實作題第二題
# 評測狀態：ZeroJudge 100 分 AC (1.0s / 256MB)
# ==============================================================================

import sys

# ------------------------------------------------------------------------------
# 📦【核心解題邏輯模組】
# ------------------------------------------------------------------------------
def solve_grid(n, m, grid):
    """
    計算 n x m 矩陣中的所有特殊位置
    回傳：(count, [(r, c), ...])
    """
    specials = []
    for i in range(n):
        for j in range(m):
            x = grid[i][j]
            s_sum = 0
            # 列偏移 dr ∈ [-x, x]
            for dr in range(-x, x + 1):
                r = i + dr
                if 0 <= r < n:
                    rem = x - abs(dr)
                    c_start = max(0, j - rem)
                    c_end = min(m - 1, j + rem)
                    # 高速切片求和
                    s_sum += sum(grid[r][c_start : c_end + 1])
            if s_sum % 10 == x % 10:
                specials.append((i, j))
    return len(specials), specials

# ------------------------------------------------------------------------------
# 🚀【ZeroJudge 複製提交專區】
# 在 ZeroJudge 送出時，請直接複製以下函式內容至主程式（支援連續多測資至 EOF）：
# ------------------------------------------------------------------------------
def run_zerojudge():
    tokens = sys.stdin.read().split()
    if not tokens:
        return
    idx = 0
    total_tokens = len(tokens)
    while idx < total_tokens:
        n = int(tokens[idx])
        m = int(tokens[idx + 1])
        idx += 2
        grid = []
        for _ in range(n):
            grid.append([int(tokens[idx + c]) for c in range(m)])
            idx += m
        
        k, res = solve_grid(n, m, grid)
        print(k)
        for r, c in res:
            print(f"{r} {c}")

# ------------------------------------------------------------------------------
# 🧪【Colab 本地自動化測試檢驗展示】
# 點擊執行 Cell 即可在 Colab 本地直接驗證官方範例與邊界測資
# ------------------------------------------------------------------------------
def run_local_tests():
    print("=" * 65)
    print("🚀 開始執行 ZeroJudge k732 本地回歸測試...")
    print("=" * 65)

    test_cases = [
        {
            "name": "官方範例一 (1x5 一維子題)",
            "n": 1, "m": 5,
            "grid": [[3, 1, 4, 5, 1]],
            "expected_count": 2,
            "expected_coords": [(0, 0), (0, 2)]
        },
        {
            "name": "官方範例二 (5x6 二維完整試題)",
            "n": 5, "m": 6,
            "grid": [
                [1, 3, 4, 1, 3, 1],
                [1, 1, 4, 1, 3, 1],
                [1, 1, 3, 2, 5, 3],
                [4, 3, 3, 1, 4, 1],
                [5, 2, 1, 1, 1, 1]
            ],
            "expected_count": 4,
            "expected_coords": [(0, 2), (1, 3), (2, 3), (3, 3)]
        },
        {
            "name": "極端邊界一 (1x1 最小單點矩陣)",
            "n": 1, "m": 1,
            "grid": [[3]],
            "expected_count": 1,
            "expected_coords": [(0, 0)]
        },
        {
            "name": "極端邊界二 (無任何符合特殊位置)",
            "n": 1, "m": 2,
            "grid": [[2, 5]],
            "expected_count": 0,
            "expected_coords": []
        }
    ]

    all_pass = True
    for case in test_cases:
        k, res = solve_grid(case["n"], case["m"], case["grid"])
        passed = (k == case["expected_count"]) and (res == case["expected_coords"])
        status = "✅ PASS" if passed else "❌ FAIL"
        if not passed:
            all_pass = False
        print(f"[{status}] {case['name']}")
        print(f"      回傳: 數量={k}, 座標={res}")
        print(f"      預期: 數量={case['expected_count']}, 座標={case['expected_coords']}")

    print("=" * 65)
    if all_pass:
        print("🎉 全部本地測試通過！代碼已達 100% 滿分 AC 標準！")
    else:
        print("⚠️ 部份測試未通過，請檢查邏輯！")
    print("=" * 65)

if __name__ == "__main__":
    # 若在 Colab 環境或手動執行時，預設展示本地自動化測試
    run_local_tests()